In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ERROR_TYPES = [
    "none",
    "wrong_model",
    "wrong_brand",
    "wrong_variant",
    "fake_clone",
    "missing_model",
    "ambiguous_variant",
    "too_generic",
    "unidentifiable",
    "not_a_phone",
    "not_an_offer",
]


In [ ]:
cwd = Path.cwd()

if (cwd / "src" / "preprocessing" / "model_keys.py").exists():
    ROOT = cwd
elif (cwd.parent / "src" / "preprocessing" / "model_keys.py").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate src/preprocessing/model_keys.py. "
        "Run this notebook from the repository root or notebooks/."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"

from src.preprocessing.model_keys import predict_model_key

print("Project root:", ROOT)
print("Model-key rules imported successfully.")


Project root: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT
Model-key rules imported successfully.


In [ ]:
# Check the input files required for manual review

required_files = [
    DATA_DIR / "processed" / "stage2"/ "stage2_val_df.csv",
    DATA_DIR / "labels" / "gold_set_Aya.csv",
]

for path in required_files:
    print("FOUND:" if path.exists() else "MISSING:", path)


FOUND: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\stage2\stage2_val_df.csv
FOUND: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\labels\gold_set_Aya.csv


In [ ]:
# Inspect the input data structure

files_to_inspect = [
    DATA_DIR / "processed" / "stage2"/ "stage2_val_df.csv",
    DATA_DIR / "labels" / "gold_set_Aya.csv",
]

for path in files_to_inspect:
    if not path.exists():
        print(path," Not ")
        continue
    df = pd.read_csv(path)
    print(f"\n{path.name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Unique ad IDs:", df["ad_id"].nunique())



stage2_val_df.csv
Shape: (1252, 60)
Columns: ['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'iphone_model', 'samsung_model', 'is_phone_listing', 'xiaomi_model', 'oppo_model', 'honor_model', 'realme_model', 'infinix_model', 'vivo_model', 'huawei_model', 'motorola_model', 'oneplus_model', 'zte_model', 'tecno_model', 'itel_model', 'google_model', 'sony_model', 'nokia_model', 'lenovo_model', 'lg_model', 'model_key', 'match_rule', 'confidence', 'family', 'storage_known', 'ram_known', 'condition_known']
Unique ad IDs: 1252

gold_set_Aya.csv
Shape

# Select validation data for manual review

In [ ]:
validation_df = pd.read_csv(DATA_DIR / "processed" / "stage2" / "stage2_val_df.csv")
gold_df = pd.read_csv(DATA_DIR / "labels" / "gold_set_Aya.csv")

validation_ids = set(validation_df["ad_id"].dropna().astype(str))
gold_ids = set(gold_df["ad_id"].dropna().astype(str))

print("Validation and Gold Set overlap:", len(validation_ids & gold_ids))


Validation and Gold Set overlap: 34


In [ ]:
# Prepare validation ads and remove examples already in the Gold Set

review_candidates = validation_df.copy()
review_candidates["ad_id"] = review_candidates["ad_id"].astype(str)
gold_ids = set(gold_df["ad_id"].dropna().astype(str))

original_rows = len(review_candidates)
review_candidates = review_candidates[
    ~review_candidates["ad_id"].isin(gold_ids)
].drop_duplicates(subset="ad_id").copy()

print("Original validation rows:", original_rows)
print("Candidates after excluding Gold Set rows:", len(review_candidates))
print("Unique candidate ads:", review_candidates["ad_id"].nunique())

display(review_candidates[["ad_id", "title", "brand"]].head())


Original validation rows: 1252
Candidates after excluding Gold Set rows: 1218
Unique candidate ads: 1218


,ad_id,title,brand
0,208029599,Honor 500 pro,honor
1,208962319,آبل - آيفون 13 برو ماكس,apple
2,208958079,سامسونج اس 24 الترا,samsung
3,207995757,Honor win rt,honor
4,207445230,Iphone 13 pro max,apple


# Apply model rules to the validation ads

In [ ]:
# Apply the existing model extraction rules

review_candidates[["predicted_model_key", "match_rule"]] = (
    review_candidates.apply(
        lambda row: predict_model_key(
            row["title"],
            row["brand"],
            with_rule=True
        ),
        axis=1,
        result_type="expand"
    )
)
display(review_candidates[[
    "ad_id",
    "title",
    "brand",
    "predicted_model_key",
    "match_rule"
]].head(10))

,ad_id,title,brand,predicted_model_key,match_rule
0,208029599,Honor 500 pro,honor,honor 500 pro,honor:rule_08
1,208962319,آبل - آيفون 13 برو ماكس,apple,apple iphone 13 pro max,apple:model_pattern_02
2,208958079,سامسونج اس 24 الترا,samsung,samsung s24 ultra,samsung:rule_14
3,207995757,Honor win rt,honor,honor win rt,honor:rule_09
4,207445230,Iphone 13 pro max,apple,apple iphone 13 pro max,apple:model_pattern_01
5,208968556,سامسونج اس 24 الترا,samsung,samsung s24 ultra,samsung:rule_14
6,208972450,Apple - iPhone 17 Pro Max - Battery 98%,apple,apple iphone 17 pro max,apple:special_case_09
7,207661107,Galaxy Z TriFold,samsung,samsung z trifold,samsung:rule_01
8,209012848,Vivo V60,vivo,vivo v60,vivo:rule_06
9,209012798,سامسونج اس 25 الترا,samsung,samsung s25 ultra,samsung:rule_14


In [ ]:
# Identify ads with unknown predictions for manual review

pred = (
    review_candidates["predicted_model_key"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

rule = (
    review_candidates["match_rule"]
    .fillna("")
    .astype(str)
    .str.strip()
)

is_unknown_prediction = pred.isin(["", "unknown"])
is_unknown_brand = rule.eq("unknown_brand")

is_unknown_known_brand = (
    is_unknown_prediction & ~is_unknown_brand
)

review_candidates["review_bucket"] = "other_rule_review"

review_candidates.loc[
    is_unknown_known_brand,
    "review_bucket"
] = "no_match_known_brand"

review_candidates.loc[
    is_unknown_brand,
    "review_bucket"
] = "unknown_brand"


print("Review bucket counts:")
display(review_candidates["review_bucket"].value_counts())

print("\nUnknown predictions:", is_unknown_prediction.sum())
print("Unknown brand rules:", is_unknown_brand.sum())
print("Unknown predictions with known brand:", is_unknown_known_brand.sum())





Review bucket counts:


review_bucket
other_rule_review       1099
unknown_brand             66
no_match_known_brand      53
Name: count, dtype: int64


Unknown predictions: 119
Unknown brand rules: 66
Unknown predictions with known brand: 53


In [ ]:
# Collect unknown and no-match cases for manual review

unknown_review = review_candidates[
    review_candidates["review_bucket"].isin([
        "no_match_known_brand",
        "unknown_brand",
    ])
].copy()

unknown_review["reviewed_model_key"] = ""
unknown_review["error_type"] = ""
unknown_review["notes"] = ""
review_df = unknown_review.copy()

print("Unknown and no-match cases:", len(review_df))
display(review_df[[
    "ad_id",
    "title",
    "brand",
    "predicted_model_key",
    "match_rule",
    "review_bucket",
]])


Unknown and no-match cases: 119


,ad_id,title,brand,predicted_model_key,match_rule,review_bucket
19,209012316,أخرىسفرة ح,NaN,unknown,unknown_brand,unknown_brand
28,207932506,الجيزه,NaN,unknown,unknown_brand,unknown_brand
53,206535330,s23 ultra,NaN,unknown,unknown_brand,unknown_brand
67,209010219,آبل - آيفون موديل اخر,apple,unknown,apple:no_match,no_match_known_brand
95,209009193,اير بودز ساوند كور ليبرتي 4 nc استعمال خفيف ح...,NaN,unknown,unknown_brand,unknown_brand
...,...,...,...,...,...,...
1198,208947123,Oppooo,oppo,unknown,oppo:no_match,no_match_known_brand
1203,208946745,Infinix Other Model,infinix,unknown,infinix:no_match,no_match_known_brand
1220,208298430,بلاك بيري بريف بجراب ذكى نادر,NaN,unknown,unknown_brand,unknown_brand
1229,208944517,ريلمى موديل اخر,realme,unknown,realme:no_match,no_match_known_brand


In [ ]:
print("Unknown and no-match ads:", len(review_df))
print("Cases by category:")
display(review_df["review_bucket"].value_counts())
display(review_df[[
    "ad_id",
    "title",
    "brand",
    "predicted_model_key",
    "match_rule",
    "review_bucket",
]].head(20))


Unknown and no-match ads: 119
Cases by category:


review_bucket
unknown_brand           66
no_match_known_brand    53
Name: count, dtype: int64

,ad_id,title,brand,predicted_model_key,match_rule,review_bucket
19,209012316,أخرىسفرة ح,NaN,unknown,unknown_brand,unknown_brand
28,207932506,الجيزه,NaN,unknown,unknown_brand,unknown_brand
53,206535330,s23 ultra,NaN,unknown,unknown_brand,unknown_brand
67,209010219,آبل - آيفون موديل اخر,apple,unknown,apple:no_match,no_match_known_brand
95,209009193,اير بودز ساوند كور ليبرتي 4 nc استعمال خفيف ح...,NaN,unknown,unknown_brand,unknown_brand
97,209009100,آبل - آيفون,apple,unknown,apple:no_match,no_match_known_brand
119,209007844,Airbudspro,NaN,unknown,unknown_brand,unknown_brand
142,209007067,infinx hot 60,NaN,unknown,unknown_brand,unknown_brand
151,209006815,pocp C75 5g,NaN,unknown,unknown_brand,unknown_brand
158,207662173,S26 ultra _ 16/1tera,NaN,unknown,unknown_brand,unknown_brand


# Review unknown-brand cases

In [ ]:
#reviewing unknown brands
unknown_brand_df = review_candidates[
    review_candidates["review_bucket"] == "unknown_brand"
].copy()

print("Number of unknown-brand cases:", len(unknown_brand_df))

display(
    unknown_brand_df[
        ["ad_id", "title", "brand", "predicted_model_key", "match_rule"]
    ].head(20)
)

Number of unknown-brand cases: 66


,ad_id,title,brand,predicted_model_key,match_rule
19,209012316,أخرىسفرة ح,NaN,unknown,unknown_brand
28,207932506,الجيزه,NaN,unknown,unknown_brand
53,206535330,s23 ultra,NaN,unknown,unknown_brand
95,209009193,اير بودز ساوند كور ليبرتي 4 nc استعمال خفيف ح...,NaN,unknown,unknown_brand
119,209007844,Airbudspro,NaN,unknown,unknown_brand
142,209007067,infinx hot 60,NaN,unknown,unknown_brand
151,209006815,pocp C75 5g,NaN,unknown,unknown_brand
158,207662173,S26 ultra _ 16/1tera,NaN,unknown,unknown_brand
168,209006075,موبايل Magic 15 max,NaN,unknown,unknown_brand
189,209005016,Mi Mix 2 | موبايل شيك ومختلف عن أي موبايل,NaN,unknown,unknown_brand


In [ ]:
no_match_df = review_candidates[
    review_candidates["review_bucket"] == "no_match_known_brand"
].copy()

print("Number of no-match cases:", len(no_match_df))

display(
    no_match_df[
        [
            "ad_id",
            "title",
            "brand",
            "predicted_model_key",
            "match_rule",
        ]
    ]
)

Number of no-match cases: 53


,ad_id,title,brand,predicted_model_key,match_rule
67,209010219,آبل - آيفون موديل اخر,apple,unknown,apple:no_match
97,209009100,آبل - آيفون,apple,unknown,apple:no_match
180,209005287,آبل - آيفون موديل اخر,apple,unknown,apple:no_match
188,209005045,ريلمى موديل اخر,realme,unknown,realme:no_match
220,209003547,آبل - آيفون موديل اخر,apple,unknown,apple:no_match
247,209002030,ريلمى موديل اخر,realme,unknown,realme:no_match
249,209001983,Infinix Other Model,infinix,unknown,infinix:no_match
292,208999403,آبل - آيفون موديل اخر,apple,unknown,apple:no_match
305,208998723,Samsung Other Model,samsung,unknown,samsung:no_match
318,208998444,ريلمى موديل اخر,realme,unknown,realme:no_match


# Find low-confidence predictions

In [ ]:
# Measure rule support using predictions on the Gold Set

gold_predictions = gold_df.apply(
    lambda row: predict_model_key(row["title"], row["brand"], with_rule=True),
    axis=1,
    result_type="expand",
)
gold_predictions.columns = ["predicted_model_key", "match_rule"]

gold_evaluation = pd.concat(
    [gold_df[["ad_id", "gold_model_key"]], gold_predictions],
    axis=1,
)
gold_evaluation = gold_evaluation[
    gold_evaluation["predicted_model_key"] != "unknown"
].copy()
gold_evaluation["is_correct"] = (
    gold_evaluation["predicted_model_key"] == gold_evaluation["gold_model_key"]
)

rule_stats = gold_evaluation.groupby("match_rule")["is_correct"].agg(
    support="size",
    correct="sum",
)
rule_stats["accuracy"] = rule_stats["correct"] / rule_stats["support"]

print("Rules used on Gold Set:", len(rule_stats))
print("Rules with fewer than 5 examples:", (rule_stats["support"] < 5).sum())
print("Gold Set accuracy on matched rows:", round(gold_evaluation["is_correct"].mean(), 3))
print("Rule support is used as a review signal; Gold Set accuracy is not an independent test score.")


Rules used on Gold Set: 65
Rules with fewer than 5 examples: 56
Gold Set accuracy on matched rows: 1.0
Rule support is used as a review signal; Gold Set accuracy is not an independent test score.


In [ ]:
# Select known-brand ads with weakly supported or generic rules

known_df = review_candidates[
    ~review_candidates["review_bucket"].isin([
        "unknown_brand",
        "no_match_known_brand",
    ])
].copy()

known_df["rule_support_in_gold"] = (
    known_df["match_rule"].map(rule_stats["support"]).fillna(0).astype(int)
)
rules = known_df["match_rule"].fillna("").astype(str)
known_df["low_conf_reason"] = ""

for condition, reason in [
    (known_df["rule_support_in_gold"].eq(0), "never_seen_in_gold"),
    (rules.str.contains(r"fallback_pattern|numeric_pattern|ambiguous"), "weak_generic_rule"),
    (rules.str.contains(r"inferred_|arabic_|mate_30_pro_5g"), "hand_patched_rule"),
]:
    known_df.loc[condition, "low_conf_reason"] += reason + "+"

known_df["low_conf_reason"] = known_df["low_conf_reason"].str.rstrip("+")
low_pool = known_df[known_df["low_conf_reason"] != ""].copy()

print("Known-brand predictions:", len(known_df))
print("Low-confidence ads:", len(low_pool))
display(low_pool["low_conf_reason"].value_counts())


Known-brand predictions: 1099
Low-confidence ads: 218


low_conf_reason
never_seen_in_gold    147
weak_generic_rule      69
hand_patched_rule       2
Name: count, dtype: int64

In [ ]:
# Prepare low-confidence ads for manual review

low_df = low_pool.copy()
low_df["review_bucket"] = "low_confidence"
low_df[["reviewed_model_key", "error_type", "notes"]] = ""

print("Low-confidence ads:", len(low_df))
print("Unique title groups:", low_df["title_group"].nunique())
display(low_df["brand"].value_counts(dropna=False))


Low-confidence ads: 218
Unique title groups: 156


brand
apple       76
honor       30
vivo        19
huawei      19
samsung     14
xiaomi      14
realme      11
infinix      8
oneplus      7
oppo         6
sony         3
motorola     3
itel         2
NaN          2
tecno        2
lenovo       1
google       1
Name: count, dtype: int64

In [ ]:
# Preview the low-confidence review cases

display(low_df[[
    "ad_id",
    "title",
    "brand",
    "predicted_model_key",
    "match_rule",
    "rule_support_in_gold",
    "low_conf_reason",
]])


,ad_id,title,brand,predicted_model_key,match_rule,rule_support_in_gold,low_conf_reason
3,207995757,Honor win rt,honor,honor win rt,honor:rule_09,0,never_seen_in_gold
7,207661107,Galaxy Z TriFold,samsung,samsung z trifold,samsung:rule_01,0,never_seen_in_gold
8,209012848,Vivo V60,vivo,vivo v60,vivo:rule_06,0,never_seen_in_gold
11,208959812,IPhone 13&14&15pro أجهزة جديدة زيرو ضمان سنة م...,apple,apple iphone 13,apple:numeric_pattern,12,weak_generic_rule
15,209012590,هونر اكس 8 بى,honor,honor x8b,honor:rule_03,0,never_seen_in_gold
...,...,...,...,...,...,...,...
1225,208944905,Huawei pura 90s pro max New جديد متبرشم,huawei,huawei pura 90s pro max,huawei:rule_01,0,never_seen_in_gold
1237,208944044,ايفون I phone Plus 7,apple,apple iphone 7 plus,apple:special_case_18,0,never_seen_in_gold
1241,208943528,سامسونج زد فليب 3,samsung,samsung z flip3,samsung:rule_05,0,never_seen_in_gold
1246,208685603,تكنو سبارك ٣٠,tecno,tecno spark 30,tecno:rule_06,0,never_seen_in_gold


# Combine review cases and save the CSV

In [ ]:
# Combine unknown, no-match, and low-confidence ads

full_review = pd.concat(
    [review_df, low_df],
    ignore_index=True,
    sort=False,
).drop_duplicates(subset="ad_id").copy()

print("Total unique ads for manual review:", len(full_review))
print("Cases by review category:")
display(full_review["review_bucket"].value_counts())


Total unique ads for manual review: 337
Cases by review category:


review_bucket
low_confidence          218
unknown_brand            66
no_match_known_brand     53
Name: count, dtype: int64

In [ ]:
# Keep the columns needed for manual review

review_columns = [
    "ad_id",
    "ad_url",
    "title",
    "brand",
    "price",
    "predicted_model_key",
    "match_rule",
    "confidence",
    "review_bucket",
    "rule_support_in_gold",
    "low_conf_reason",
    "reviewed_model_key",
    "error_type",
    "notes",
]

for column in review_columns:
    if column not in full_review.columns:
        full_review[column] = ""

for column in ["reviewed_model_key", "error_type", "notes"]:
    full_review[column] = full_review[column].fillna("")

# Convert relative Dubizzle paths into clickable listing links.
relative_url = full_review["ad_url"].fillna("").astype(str).str.startswith("/")
full_review.loc[relative_url, "ad_url"] = (
    "https://www.dubizzle.com.eg" + full_review.loc[relative_url, "ad_url"].astype(str)
)

full_review = full_review[review_columns].sort_values(
    ["review_bucket", "brand", "ad_id"],
    na_position="last",
).reset_index(drop=True)

display(full_review.head(10))


,ad_id,ad_url,title,brand,price,predicted_model_key,match_rule,confidence,review_bucket,rule_support_in_gold,low_conf_reason,reviewed_model_key,error_type,notes
0,199959008,https://www.dubizzle.com.eg/ad/iphone-se-2022-...,iPhone Se 2022 ( 3rd generation),apple,20000.0,apple iphone se,apple:special_model_05,NaN,low_confidence,0.0,never_seen_in_gold,,,
1,202218973,https://www.dubizzle.com.eg/ad/%D8%AA%D9%84%D9...,تليفون ايفون 12 مبنى Apple,apple,12500.0,apple iphone 12,apple:numeric_pattern,1.0,low_confidence,12.0,weak_generic_rule,,,
2,206895716,https://www.dubizzle.com.eg/ad/iphone-15-%D9%8...,iphone 15 معفي من الضريبه,apple,42000.0,apple iphone 15,apple:numeric_pattern,1.0,low_confidence,12.0,weak_generic_rule,,,
3,207654180,https://www.dubizzle.com.eg/ad/%D8%A7%D9%8A%D9...,ايفون ٧ فاصل بصمة ١٢٨ شغال كويس,apple,3000.0,apple iphone 7,apple:numeric_pattern,1.0,low_confidence,12.0,weak_generic_rule,,,
4,208029051,https://www.dubizzle.com.eg/ad/iphone-13-used-...,iPhone 13 - Used,apple,27000.0,apple iphone 13,apple:numeric_pattern,1.0,low_confidence,12.0,weak_generic_rule,,,
5,208106958,https://www.dubizzle.com.eg/ad/%D8%A7%D9%8A%D9...,ايفون ١١ للبدل ب اعلي,apple,12000.0,apple iphone 11,apple:numeric_pattern,1.0,low_confidence,12.0,weak_generic_rule,,,
6,208266521,https://www.dubizzle.com.eg/ad/apple-iphone-6-...,Apple - iPhone 6,apple,2000.0,apple iphone 6,apple:numeric_pattern,1.0,low_confidence,12.0,weak_generic_rule,,,
7,208443403,https://www.dubizzle.com.eg/ad/%D8%A7%D9%8A%D9...,ايفون 6,apple,2000.0,apple iphone 6,apple:numeric_pattern,1.0,low_confidence,12.0,weak_generic_rule,,,
8,208450623,https://www.dubizzle.com.eg/ad/%D8%A2%D8%A8%D9...,آبل - آيفون 14,apple,28000.0,apple iphone 14,apple:fallback_pattern,1.0,low_confidence,14.0,weak_generic_rule,,,
9,208460025,https://www.dubizzle.com.eg/ad/apple-iphone-12...,Apple - iPhone 12 mini 128GB,apple,11000.0,apple iphone 12 mini,apple:numeric_pattern,1.0,low_confidence,12.0,weak_generic_rule,,,


In [ ]:
# Save the CSV in the project data folder

OUTPUT_PATH = DATA_DIR / "labels" / "manual_review_candidates.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
full_review.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved review file:", OUTPUT_PATH)
print("Rows:", len(full_review))


Saved review file: e:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\labels\manual_review_candidates.csv
Rows: 337


### Manual review instructions

Open `data/labels/manual_review_candidates.csv` in Excel. Fill `reviewed_model_key`, `error_type`, and `notes`. Save a copy as `manual_review_candidates_reviewed.csv` in the same folder, then run the analysis cells below.

Use one `error_type` from the list defined in the first cell. Keep the final test set out of this manual review.


In [ ]:
REVIEWED_CSV = DATA_DIR / "labels" / "manual_review_candidates_reviewed.csv"

if not REVIEWED_CSV.exists():
    print("Manual review is not complete yet.")
    print("Save the completed Excel file as:", REVIEWED_CSV)
    reviewed = None
else:
    reviewed = pd.read_csv(REVIEWED_CSV, dtype={"ad_id": str}, encoding="utf-8-sig")
    for column in ["reviewed_model_key", "predicted_model_key", "error_type"]:
        reviewed[column] = reviewed[column].astype("string").str.strip().str.lower()

    missing_model = reviewed["reviewed_model_key"].isna() | reviewed["reviewed_model_key"].eq("")
    missing_type = reviewed["error_type"].isna() | reviewed["error_type"].eq("")
    invalid_type = ~reviewed["error_type"].isin(ERROR_TYPES) & ~missing_type

    print(f"Model labels completed: {(~missing_model).sum()} / {len(reviewed)}")
    print(f"Error types completed: {(~missing_type).sum()} / {len(reviewed)}")
    print(f"Invalid error types: {invalid_type.sum()}")

    if missing_model.any():
        print("Rows still needing a model label or an explicit unknown value:")
        display(reviewed.loc[missing_model, ["ad_id", "title", "error_type", "notes"]])


Model labels completed: 195 / 205
Error types completed: 205 / 205
Invalid error types: 0
Rows still needing a model label or an explicit unknown value:


,ad_id,title,error_type,notes
28,208971482,Apple - iPhone 17,ambiguous_variant,generic auto-title; price 65000 could be 17 Pr...
31,208973805,Apple - iPhone 17,ambiguous_variant,generic auto-title; price 65000 could be 17 Pr...
35,208979846,Apple - iPhone 17,ambiguous_variant,generic auto-title; price 70000 could be 17 Pr...
38,208986186,Apple - iPhone 17,ambiguous_variant,generic auto-title; price 72000 could be 17 Pr...
39,208986305,Apple - iPhone 17,ambiguous_variant,generic auto-title; price 70000 could be 17 Pr...
47,208995410,Apple - iPhone 17,ambiguous_variant,generic auto-title; price 85000 could be 17 Pr...
51,209002174,Apple - iPhone 17,ambiguous_variant,generic auto-title; price 65000 could be 17 Pr...
117,206257717,شاشه touchscreen panel sony xperia Z Ultra للبيع,not_a_phone,touchscreen panel (spare part) for Xperia Z Ul...
170,201162683,matepad 10.4 wifi + 4g,not_a_phone,Huawei MatePad 10.4 is a tablet (huawei matepa...
191,208971638,محتاج تلفون كويس شاشه كيرف شرط حاله جيده,not_an_offer,"buyer request (""need a good phone with curved ..."


In [ ]:
if reviewed is None:
    print("Run this summary after adding the reviewed CSV.")
else:
    print("--- Review categories ---")
    display(reviewed["error_type"].value_counts(dropna=False).rename_axis("error_type").to_frame("rows"))

    labeled = reviewed[
        reviewed["reviewed_model_key"].notna()
        & reviewed["reviewed_model_key"].ne("")
    ].copy()
    labeled["model_key_matches"] = (
        labeled["predicted_model_key"] == labeled["reviewed_model_key"]
    )

    print(f"Rows with a reviewed model key: {len(labeled)} / {len(reviewed)}")
    if len(labeled):
        print(
            "Prediction agreement on labeled rows:",
            f"{labeled['model_key_matches'].mean():.2%}",
        )

    error_by_brand = pd.crosstab(
        reviewed["brand"].fillna("NO_BRAND"),
        reviewed["error_type"],
    )
    display(error_by_brand.assign(total=error_by_brand.sum(axis=1)).sort_values("total", ascending=False))


--- Review categories ---


,rows
error_type,
none,123
too_generic,31
missing_model,21
unidentifiable,13
ambiguous_variant,9
not_a_phone,2
wrong_model,2
fake_clone,1
wrong_variant,1


Rows with a reviewed model key: 195 / 205
Prediction agreement on labeled rows: 86.15%


error_type,ambiguous_variant,fake_clone,missing_model,none,not_a_phone,not_an_offer,too_generic,unidentifiable,wrong_brand,wrong_model,wrong_variant,total
brand,,,,,,,,,,,,
apple,8,1,0,56,0,0,0,0,0,0,0,65
NO_BRAND,1,0,21,0,1,1,0,12,0,0,0,36
xiaomi,0,0,0,9,0,0,21,0,1,0,0,31
honor,0,0,0,13,0,0,0,0,0,2,0,15
samsung,0,0,0,7,0,0,0,1,0,0,1,9
infinix,0,0,0,3,0,0,5,0,0,0,0,8
oppo,0,0,0,3,0,0,4,0,0,0,0,7
realme,0,0,0,7,0,0,0,0,0,0,0,7
huawei,0,0,0,7,0,0,0,0,0,0,0,7


In [ ]:
if reviewed is None:
    print("Add the reviewed CSV before listing unresolved model cases.")
else:
    missing_model = reviewed["reviewed_model_key"].isna() | reviewed["reviewed_model_key"].eq("")
    if missing_model.any():
        print("Rows without a reviewed model key:")
        display(reviewed.loc[missing_model, [
            "ad_id", "title", "predicted_model_key", "error_type", "notes"
        ]])

    labeled = reviewed[~missing_model].copy()
    labeled["exact_match"] = labeled["predicted_model_key"] == labeled["reviewed_model_key"]
    print(f"Exact matches among labeled rows: {labeled['exact_match'].sum()} / {len(labeled)}")
    display(labeled.loc[~labeled["exact_match"], [
        "title", "predicted_model_key", "reviewed_model_key", "match_rule", "error_type"
    ]])


Rows without a reviewed model key:


,ad_id,title,predicted_model_key,error_type,notes
28,208971482,Apple - iPhone 17,apple iphone 17,ambiguous_variant,generic auto-title; price 65000 could be 17 Pr...
31,208973805,Apple - iPhone 17,apple iphone 17,ambiguous_variant,generic auto-title; price 65000 could be 17 Pr...
35,208979846,Apple - iPhone 17,apple iphone 17,ambiguous_variant,generic auto-title; price 70000 could be 17 Pr...
38,208986186,Apple - iPhone 17,apple iphone 17,ambiguous_variant,generic auto-title; price 72000 could be 17 Pr...
39,208986305,Apple - iPhone 17,apple iphone 17,ambiguous_variant,generic auto-title; price 70000 could be 17 Pr...
47,208995410,Apple - iPhone 17,apple iphone 17,ambiguous_variant,generic auto-title; price 85000 could be 17 Pr...
51,209002174,Apple - iPhone 17,apple iphone 17,ambiguous_variant,generic auto-title; price 65000 could be 17 Pr...
117,206257717,شاشه touchscreen panel sony xperia Z Ultra للبيع,sony xperia z ultra,not_a_phone,touchscreen panel (spare part) for Xperia Z Ul...
170,201162683,matepad 10.4 wifi + 4g,unknown,not_a_phone,Huawei MatePad 10.4 is a tablet (huawei matepa...
191,208971638,محتاج تلفون كويس شاشه كيرف شرط حاله جيده,unknown,not_an_offer,"buyer request (""need a good phone with curved ..."


Exact matches among labeled rows: 168 / 195


,title,predicted_model_key,reviewed_model_key,match_rule,error_type
7,ايفون 17 ماكس صغير نظام اندرويد,apple iphone 17,unknown,apple:numeric_pattern,fake_clone
75,بيع هونور 8Xمستعمل حاله جيده,honor x8,honor 8x,honor:rule_04,wrong_model
78,Honor 8 Pro,honor 8 pro,honor magic 8 pro,honor:rule_14,wrong_model
110,سامسونج J3.2016,samsung j3,samsung j3 2016,samsung:rule_22,wrong_variant
127,Xiaomi Redmi c25y,xiaomi redmi c25y,realme c25y,xiaomi:rule_05,wrong_brand
136,ZTE Other Model,zte unknown,unknown,zte:rule_08,too_generic
169,P50 Pocket,unknown,huawei p50 pocket,unknown_brand,missing_model
171,GT6 5G Dual SIM Fluid Silver 12GB RAM 256GB 5G...,unknown,realme gt 6,unknown_brand,missing_model
172,Asus phone ROG 5,unknown,asus rog phone 5,unknown_brand,missing_model
173,s24 ultra gray titanium,unknown,samsung s24 ultra,unknown_brand,missing_model
